# 中文 NER 与实体链接：从 span 到 tenant-scoped KB

这份 notebook 实现一条离线可运行的工程链路：

原始文本 → span 检测 → 重叠消解 → BIO/BILOU 导出 → 候选生成 → 上下文排序 → 歧义拒识 → 带 provenance 的实体链接结果

规则与词典实现用于讲清 offset、候选和服务合同，不能冒充通用 NER/EL 模型。受控数据上的高分只说明回归样例通过。

## 1. 先冻结 API 与数据合同

输入：doc_id、doc_version、tenant_id（来自鉴权声明）、raw_text、requested_kb_version。

输出中的 mention 使用半开区间 [start, end)，并至少返回 surface、entity_type、entity_id 或 ABSTAIN、confidence、candidate_ids、extractor_version、kb_version、evidence。

raw_text 不可被规范化结果覆盖。Python 字符索引、UTF-8 byte offset、JavaScript UTF-16 code unit 不相同；跨语言 API 必须声明 offset_unit。本例使用 Unicode code point。

In [ ]:
from dataclasses import dataclass, asdict  # 导入本单元所需的依赖。
from collections import defaultdict  # 导入本单元所需的依赖。
import hashlib  # 导入本单元所需的依赖。
import json  # 导入本单元所需的依赖。
import math  # 导入本单元所需的依赖。
import re  # 导入本单元所需的依赖。

EXTRACTOR_VERSION = 'dict-ner-v1.1'  # 计算并保存当前步骤的中间状态。
LINKER_VERSION = 'context-linker-v1.0'  # 计算并保存当前步骤的中间状态。
OFFSET_UNIT = 'unicode_code_point'  # 计算并保存当前步骤的中间状态。

@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Entity:  # 定义承载本节状态与行为的数据结构。
    entity_id: str  # 执行当前语句以推进本节示例。
    tenant_id: str  # 执行当前语句以推进本节示例。
    canonical_name: str  # 执行当前语句以推进本节示例。
    aliases: tuple  # 执行当前语句以推进本节示例。
    entity_type: str  # 执行当前语句以推进本节示例。
    keywords: tuple  # 执行当前语句以推进本节示例。
    active: bool  # 执行当前语句以推进本节示例。
    kb_version: int  # 执行当前语句以推进本节示例。

entities = [  # 计算并保存当前步骤的中间状态。
    Entity('org:pku', 'tenant-a', '北京大学', ('北京大学', '北大'), 'ORG',  # 执行当前语句以推进本节示例。
           ('高校', '大学', '院系', '学生', '科研'), True, 3),  # 执行当前语句以推进本节示例。
    Entity('org:pku-hospital', 'tenant-a', '北京大学第一医院', ('北京大学第一医院', '北大医院'), 'ORG',  # 执行当前语句以推进本节示例。
           ('医院', '医学', '门诊', '医生', '患者'), True, 3),  # 执行当前语句以推进本节示例。
    Entity('per:zhang-wei-01', 'tenant-a', '张伟（医生）', ('张伟',), 'PER',  # 执行当前语句以推进本节示例。
           ('医生', '医学', '医院', '门诊'), True, 3),  # 执行当前语句以推进本节示例。
    Entity('company:apple', 'tenant-a', 'Apple Inc.', ('苹果', 'Apple'), 'ORG',  # 执行当前语句以推进本节示例。
           ('公司', '手机', 'iphone', 'mac', '芯片', '发布'), True, 3),  # 执行当前语句以推进本节示例。
    Entity('concept:apple-fruit', 'tenant-a', '苹果（水果）', ('苹果',), 'PRODUCT',  # 执行当前语句以推进本节示例。
           ('水果', '甜', '食用', '品种', '果树'), True, 3),  # 执行当前语句以推进本节示例。
    Entity('project:beida', 'tenant-b', '北大迁移项目', ('北大',), 'PROJECT',  # 执行当前语句以推进本节示例。
           ('迁移', '内部', '项目'), True, 8),  # 执行当前语句以推进本节示例。
    Entity('org:old', 'tenant-a', '旧机构', ('旧机构',), 'ORG',  # 执行当前语句以推进本节示例。
           ('停用',), False, 2),  # 执行当前语句以推进本节示例。
]  # 执行当前语句以推进本节示例。
assert len({(e.tenant_id, e.entity_id, e.kb_version) for e in entities}) == len(entities)  # 用受控断言验证关键不变量。
print('KB fixture entities:', len(entities))  # 执行当前语句以推进本节示例。

## 2. BIO、BILOU 与 span 的取舍

BIO 给每个 token 标 B/I/O；BILOU 进一步区分单 token 的 U 与多 token 末尾 L，通常能更明确表达边界。工程存储更适合 span + type，因为它不绑定某个 tokenizer。

标签转换必须在 tokenizer 固定后进行。中文按字只是教学选择；生产模型可能使用 subword，需要保存 char↔token 对齐。嵌套/重叠实体无法放入单层 BIO 序列，应保留 span 列表或多层标签，而不是静默丢弃。

In [ ]:
def validate_non_overlapping_spans(text, spans):  # 定义本节可复用的核心函数。
    previous_end = 0  # 计算并保存当前步骤的中间状态。
    for start, end, entity_type in sorted(spans):  # 遍历输入元素以累积或检查结果。
        if not (0 <= start < end <= len(text)):  # 按当前条件选择后续控制路径。
            raise ValueError('span_out_of_range')  # 遇到非法合同立即显式失败。
        if start < previous_end:  # 按当前条件选择后续控制路径。
            raise ValueError('overlapping_spans_need_multiple_layers')  # 遇到非法合同立即显式失败。
        previous_end = end  # 计算并保存当前步骤的中间状态。

def spans_to_bilou(text, spans):  # 定义本节可复用的核心函数。
    validate_non_overlapping_spans(text, spans)  # 执行当前语句以推进本节示例。
    labels = ['O'] * len(text)  # 计算并保存当前步骤的中间状态。
    for start, end, entity_type in spans:  # 遍历输入元素以累积或检查结果。
        width = end - start  # 计算并保存当前步骤的中间状态。
        if width == 1:  # 按当前条件选择后续控制路径。
            labels[start] = f'U-{entity_type}'  # 计算并保存当前步骤的中间状态。
        else:  # 处理前置条件不成立的分支。
            labels[start] = f'B-{entity_type}'  # 计算并保存当前步骤的中间状态。
            for i in range(start + 1, end - 1):  # 遍历输入元素以累积或检查结果。
                labels[i] = f'I-{entity_type}'  # 计算并保存当前步骤的中间状态。
            labels[end - 1] = f'L-{entity_type}'  # 计算并保存当前步骤的中间状态。
    return labels  # 返回当前分支计算出的结果。

sample = '张伟就职于北大医院'  # 计算并保存当前步骤的中间状态。
sample_spans = [(0, 2, 'PER'), (5, 9, 'ORG')]  # 计算并保存当前步骤的中间状态。
labels = spans_to_bilou(sample, sample_spans)  # 计算并保存当前步骤的中间状态。
assert labels[:2] == ['B-PER', 'L-PER']  # 用受控断言验证关键不变量。
assert labels[5:9] == ['B-ORG', 'I-ORG', 'I-ORG', 'L-ORG']  # 用受控断言验证关键不变量。
print(list(zip(sample, labels)))  # 执行当前语句以推进本节示例。

## 3. Offset 是跨服务协议的一部分

Python len('😀') 为 1，而 JavaScript UTF-16 长度为 2；UTF-8 中又占 4 bytes。若标注平台、模型服务、搜索索引各用不同单位，emoji 前后的实体会整体漂移。

建议响应同时给 raw_text_sha256 与 offset_unit；消费者先校验文本 hash。下面显式转换 code-point span 到 UTF-8 byte span，避免把字符索引直接当 byte index。

In [ ]:
def codepoint_to_utf8_span(text, start, end):  # 定义本节可复用的核心函数。
    if not (0 <= start <= end <= len(text)):  # 按当前条件选择后续控制路径。
        raise ValueError('invalid_span')  # 遇到非法合同立即显式失败。
    byte_start = len(text[:start].encode('utf-8'))  # 计算并保存当前步骤的中间状态。
    byte_end = len(text[:end].encode('utf-8'))  # 计算并保存当前步骤的中间状态。
    return byte_start, byte_end  # 返回当前分支计算出的结果。

emoji_text = '😀张伟在北大'  # 计算并保存当前步骤的中间状态。
cp_span = (1, 3)  # 计算并保存当前步骤的中间状态。
byte_span = codepoint_to_utf8_span(emoji_text, *cp_span)  # 计算并保存当前步骤的中间状态。
assert emoji_text[cp_span[0]:cp_span[1]] == '张伟'  # 用受控断言验证关键不变量。
assert emoji_text.encode('utf-8')[byte_span[0]:byte_span[1]].decode() == '张伟'  # 用受控断言验证关键不变量。
assert byte_span[0] == 4  # 用受控断言验证关键不变量。
print({'code_point_span': cp_span, 'utf8_byte_span': byte_span})  # 执行当前语句以推进本节示例。

## 4. 中文规则/词典 NER：保留所有命中

中文别名不一定有空格，先用每个 tenant 的 alias 表做确定性查找。实现返回所有命中，因此“北大医院”中既可能命中“北大”，也命中更长的“北大医院”。

线上词典通常用 Trie/Aho–Corasick 降低扫描成本，并为大小写、全半角、别名来源分别版本化。规范化只能用于候选查找，最终 span 必须落回 raw text。尤其是 casefold 可能把 `ß` 展开成 `ss`，不能把折叠字符串的位置直接当 raw offset；下面为每个折叠字符保存 raw code-point 来源，并用 `Straße` 反例验证。

In [ ]:
def select_snapshot(entity_rows, tenant_id, kb_version):  # 定义本节可复用的核心函数。
    # 每个 entity 选择 <= 请求版本的最新 revision；未修改实体不会在 KB 升版时消失。
    latest = {}  # 计算并保存当前步骤的中间状态。
    for entity in entity_rows:  # 遍历输入元素以累积或检查结果。
        if entity.tenant_id != tenant_id or entity.kb_version > kb_version:  # 按当前条件选择后续控制路径。
            continue  # 调整当前循环或占位控制流。
        previous = latest.get(entity.entity_id)  # 计算并保存当前步骤的中间状态。
        if previous is None or entity.kb_version > previous.kb_version:  # 按当前条件选择后续控制路径。
            latest[entity.entity_id] = entity  # 计算并保存当前步骤的中间状态。
    return sorted((entity for entity in latest.values() if entity.active),  # 返回当前分支计算出的结果。
                  key=lambda entity: entity.entity_id)  # 计算并保存当前步骤的中间状态。

def build_alias_index(entity_rows, tenant_id, kb_version):  # 定义本节可复用的核心函数。
    index = defaultdict(list)  # 计算并保存当前步骤的中间状态。
    for entity in select_snapshot(entity_rows, tenant_id, kb_version):  # 遍历输入元素以累积或检查结果。
        for alias in entity.aliases:  # 遍历输入元素以累积或检查结果。
            index[alias.casefold()].append(entity.entity_id)  # 执行当前语句以推进本节示例。
    return {alias: sorted(set(ids)) for alias, ids in index.items()}  # 返回当前分支计算出的结果。

def casefold_with_raw_map(text):  # 定义本节可复用的核心函数。
    folded_chars, folded_to_raw = [], []  # 计算并保存当前步骤的中间状态。
    for raw_index, ch in enumerate(text):  # 遍历输入元素以累积或检查结果。
        piece = ch.casefold()  # 计算并保存当前步骤的中间状态。
        folded_chars.extend(piece)  # 执行当前语句以推进本节示例。
        folded_to_raw.extend([raw_index] * len(piece))  # 执行当前语句以推进本节示例。
    folded = ''.join(folded_chars)  # 计算并保存当前步骤的中间状态。
    if folded != text.casefold():  # 按当前条件选择后续控制路径。
        raise ValueError('non_local_casefold_requires_range_mapping')  # 遇到非法合同立即显式失败。
    return folded, folded_to_raw  # 返回当前分支计算出的结果。

def find_all_mentions(text, alias_index):  # 定义本节可复用的核心函数。
    found = []  # 计算并保存当前步骤的中间状态。
    folded, folded_to_raw = casefold_with_raw_map(text)  # 计算并保存当前步骤的中间状态。
    for alias, entity_ids in alias_index.items():  # 遍历输入元素以累积或检查结果。
        start = 0  # 计算并保存当前步骤的中间状态。
        while True:  # 在终止条件满足前持续推进状态。
            pos = folded.find(alias, start)  # 计算并保存当前步骤的中间状态。
            if pos < 0:  # 按当前条件选择后续控制路径。
                break  # 调整当前循环或占位控制流。
            raw_positions = folded_to_raw[pos:pos + len(alias)]  # 计算并保存当前步骤的中间状态。
            raw_start, raw_end = min(raw_positions), max(raw_positions) + 1  # 计算并保存当前步骤的中间状态。
            found.append({  # 执行当前语句以推进本节示例。
                'start': raw_start, 'end': raw_end,  # 执行当前语句以推进本节示例。
                'surface': text[raw_start:raw_end],  # 执行当前语句以推进本节示例。
                'alias_key': alias,  # 执行当前语句以推进本节示例。
                'candidate_ids': tuple(sorted(entity_ids)),  # 执行当前语句以推进本节示例。
            })  # 执行当前语句以推进本节示例。
            start = pos + 1  # 计算并保存当前步骤的中间状态。
    return sorted(found, key=lambda m: (m['start'], -(m['end']-m['start']), m['alias_key']))  # 返回当前分支计算出的结果。

alias_a = build_alias_index(entities, 'tenant-a', 3)  # 计算并保存当前步骤的中间状态。
all_mentions = find_all_mentions('张伟在北大医院工作', alias_a)  # 计算并保存当前步骤的中间状态。
german_mentions = find_all_mentions('X Straße Y', {'strasse': ('concept:street',)})  # 计算并保存当前步骤的中间状态。
assert any(m['surface'] == '北大' for m in all_mentions)  # 用受控断言验证关键不变量。
assert any(m['surface'] == '北大医院' for m in all_mentions)  # 用受控断言验证关键不变量。
assert alias_a['苹果'] == ['company:apple', 'concept:apple-fruit']  # 用受控断言验证关键不变量。
assert german_mentions[0]['surface'] == 'Straße'  # 用受控断言验证关键不变量。
assert (german_mentions[0]['start'], german_mentions[0]['end']) == (2, 8)  # 用受控断言验证关键不变量。
print(all_mentions, german_mentions)  # 执行当前语句以推进本节示例。

## 5. 重叠实体不能靠字典迭代顺序决定

搜索/链接接口常需要一层不重叠 mention，可采用 longest-leftmost、模型分数最大化或领域约束；但原始 all_mentions 应保留用于审计和嵌套实体任务。

这里的 longest-leftmost：先按起点，优先更长 span；接受后跳过与已接受 span 重叠的候选。它会在“北大医院”中选择医院而非“北大”，这是显式策略，不是普遍正确答案。

In [ ]:
def overlaps(a, b):  # 定义本节可复用的核心函数。
    return a['start'] < b['end'] and b['start'] < a['end']  # 返回当前分支计算出的结果。

def longest_leftmost(mentions):  # 定义本节可复用的核心函数。
    selected = []  # 计算并保存当前步骤的中间状态。
    for mention in sorted(mentions, key=lambda m: (m['start'], -(m['end']-m['start']), m['surface'])):  # 遍历输入元素以累积或检查结果。
        if not any(overlaps(mention, kept) for kept in selected):  # 按当前条件选择后续控制路径。
            selected.append(mention)  # 执行当前语句以推进本节示例。
    return sorted(selected, key=lambda m: m['start'])  # 返回当前分支计算出的结果。

selected = longest_leftmost(all_mentions)  # 计算并保存当前步骤的中间状态。
assert [m['surface'] for m in selected] == ['张伟', '北大医院']  # 用受控断言验证关键不变量。
assert not any(overlaps(a, b) for i, a in enumerate(selected) for b in selected[i+1:])  # 用受控断言验证关键不变量。
print(selected)  # 执行当前语句以推进本节示例。

## 6. Candidate generation：高召回，但必须 tenant-scoped

候选可来自 exact alias、归一化 alias、拼音/缩写、搜索索引或向量 ANN。candidate recall 是独立指标：gold entity 若没进入候选，后续 ranker 无法补救。

安全边界比排序更早：tenant 与 KB snapshot 先过滤，再生成候选。不能从全局候选中打低分，因为 entity_id、别名、数量本身就可能泄密。

In [ ]:
def generate_candidates(surface, tenant_id, kb_version, limit=10, entity_rows=entities):  # 定义本节可复用的核心函数。
    snapshot = select_snapshot(entity_rows, tenant_id, kb_version)  # 计算并保存当前步骤的中间状态。
    entity_by_id = {entity.entity_id: entity for entity in snapshot}  # 计算并保存当前步骤的中间状态。
    alias_index = build_alias_index(snapshot, tenant_id, kb_version)  # 计算并保存当前步骤的中间状态。
    ids = alias_index.get(surface.casefold(), [])  # 计算并保存当前步骤的中间状态。
    return [entity_by_id[entity_id] for entity_id in ids[:limit]]  # 返回当前分支计算出的结果。

apple_candidates = generate_candidates('苹果', 'tenant-a', 3)  # 计算并保存当前步骤的中间状态。
assert {e.entity_id for e in apple_candidates} == {'company:apple', 'concept:apple-fruit'}  # 用受控断言验证关键不变量。
assert generate_candidates('北大', 'tenant-b', 3) == []  # 用受控断言验证关键不变量。
assert [e.entity_id for e in generate_candidates('北大', 'tenant-b', 8)] == ['project:beida']  # 用受控断言验证关键不变量。
print([asdict(e) for e in apple_candidates])  # 执行当前语句以推进本节示例。

## 7. Context ranking 与歧义拒识

ranker 可以使用 mention 左右窗口、文档主题、实体描述、类型先验、流行度和图邻居。流行度很强但会吞掉长尾实体，必须与上下文证据分开记录。

教学 ranker 对领域关键词做精确/ASCII token 命中，并使用很小的 alias prior。只有 top score 达阈值且 top1-top2 margin 足够才链接；否则 ABSTAIN。置信度不是天然概率，线上需要在独立验证集按桶校准。

In [ ]:
def context_terms(text, candidates):  # 定义本节可复用的核心函数。
    ascii_terms = set(re.findall(r'[a-z0-9]+', text.casefold()))  # 计算并保存当前步骤的中间状态。
    # 只读取已通过 tenant/snapshot 过滤的候选关键词，避免全局 KB 特征越权。
    chinese_terms = {term for entity in candidates for term in entity.keywords if term in text}  # 计算并保存当前步骤的中间状态。
    return ascii_terms | chinese_terms  # 返回当前分支计算出的结果。

def rank_candidates(surface, context, candidates):  # 定义本节可复用的核心函数。
    terms = context_terms(context, candidates)  # 计算并保存当前步骤的中间状态。
    ranked = []  # 计算并保存当前步骤的中间状态。
    for entity in candidates:  # 遍历输入元素以累积或检查结果。
        keyword_hits = sorted(terms & {k.casefold() for k in entity.keywords})  # 计算并保存当前步骤的中间状态。
        canonical_bonus = 0.1 if entity.canonical_name.casefold() in context.casefold() else 0.0  # 计算并保存当前步骤的中间状态。
        score = 1.0 * len(keyword_hits) + canonical_bonus  # 计算并保存当前步骤的中间状态。
        ranked.append({  # 执行当前语句以推进本节示例。
            'entity_id': entity.entity_id,  # 执行当前语句以推进本节示例。
            'entity_type': entity.entity_type,  # 执行当前语句以推进本节示例。
            'score': score,  # 执行当前语句以推进本节示例。
            'evidence': {'keyword_hits': keyword_hits, 'canonical_bonus': canonical_bonus},  # 执行当前语句以推进本节示例。
        })  # 执行当前语句以推进本节示例。
    return sorted(ranked, key=lambda row: (-row['score'], row['entity_id']))  # 返回当前分支计算出的结果。

def choose_or_abstain(ranked, min_score=1.0, min_margin=0.5):  # 定义本节可复用的核心函数。
    if not ranked or ranked[0]['score'] < min_score:  # 按当前条件选择后续控制路径。
        return {'entity_id': None, 'entity_type': None, 'reason': 'insufficient_evidence', 'ranked': ranked}  # 返回当前分支计算出的结果。
    second = ranked[1]['score'] if len(ranked) > 1 else float('-inf')  # 计算并保存当前步骤的中间状态。
    if ranked[0]['score'] - second < min_margin:  # 按当前条件选择后续控制路径。
        return {'entity_id': None, 'entity_type': None, 'reason': 'ambiguous_margin', 'ranked': ranked}  # 返回当前分支计算出的结果。
    return {'entity_id': ranked[0]['entity_id'], 'entity_type': ranked[0]['entity_type'],  # 返回当前分支计算出的结果。
            'reason': 'linked', 'ranked': ranked}  # 执行当前语句以推进本节示例。

company = choose_or_abstain(rank_candidates('苹果', '苹果公司发布新款 iPhone 手机', apple_candidates))  # 计算并保存当前步骤的中间状态。
fruit = choose_or_abstain(rank_candidates('苹果', '这种苹果是很甜的水果品种', apple_candidates))  # 计算并保存当前步骤的中间状态。
ambiguous = choose_or_abstain(rank_candidates('苹果', '我喜欢苹果', apple_candidates))  # 计算并保存当前步骤的中间状态。
assert company['entity_id'] == 'company:apple' and company['entity_type'] == 'ORG'  # 用受控断言验证关键不变量。
assert fruit['entity_id'] == 'concept:apple-fruit' and fruit['entity_type'] == 'PRODUCT'  # 用受控断言验证关键不变量。
assert ambiguous['entity_id'] is None and ambiguous['entity_type'] is None  # 用受控断言验证关键不变量。
print(company, fruit, ambiguous, sep='\n')  # 计算并保存当前步骤的中间状态。

## 8. 端到端链接与 provenance

每条输出保存 doc/kb/extractor/linker 版本、raw text hash、候选列表和证据特征。只保存最终 entity_id 会让错误无法归因：究竟是 NER 漏掉、候选没召回、排序错，还是阈值拒识？

API 中 confidence 这里用简单 margin proxy，仅为 trace；不承诺是概率。生产应由验证集校准，并在模型或 KB 更新后重新标定。

In [ ]:
def link_document(doc_id, doc_version, tenant_id, text, kb_version, entity_rows=entities):  # 定义本节可复用的核心函数。
    # mention index、candidate metadata 与 ranker 必须读取同一份 snapshot rows。
    alias_index = build_alias_index(entity_rows, tenant_id, kb_version)  # 计算并保存当前步骤的中间状态。
    mentions = longest_leftmost(find_all_mentions(text, alias_index))  # 计算并保存当前步骤的中间状态。
    outputs = []  # 计算并保存当前步骤的中间状态。
    for mention in mentions:  # 遍历输入元素以累积或检查结果。
        candidates = generate_candidates(mention['surface'], tenant_id, kb_version,  # 计算并保存当前步骤的中间状态。
                                         entity_rows=entity_rows)  # 计算并保存当前步骤的中间状态。
        ranked = rank_candidates(mention['surface'], text, candidates)  # 计算并保存当前步骤的中间状态。
        decision = choose_or_abstain(ranked)  # 计算并保存当前步骤的中间状态。
        margin = 0.0  # 计算并保存当前步骤的中间状态。
        if ranked:  # 按当前条件选择后续控制路径。
            margin = ranked[0]['score'] - (ranked[1]['score'] if len(ranked) > 1 else 0.0)  # 计算并保存当前步骤的中间状态。
        outputs.append({  # 执行当前语句以推进本节示例。
            'start': mention['start'], 'end': mention['end'],  # 执行当前语句以推进本节示例。
            'surface': mention['surface'], 'offset_unit': OFFSET_UNIT,  # 执行当前语句以推进本节示例。
            'entity_id': decision['entity_id'], 'entity_type': decision['entity_type'],  # 执行当前语句以推进本节示例。
            'decision_reason': decision['reason'],  # 执行当前语句以推进本节示例。
            'confidence_proxy': round(max(0.0, min(1.0, margin / 2)), 3),  # 执行当前语句以推进本节示例。
            'candidate_ids': [row['entity_id'] for row in ranked],  # 执行当前语句以推进本节示例。
            'evidence': ranked[0]['evidence'] if ranked else {},  # 执行当前语句以推进本节示例。
            'provenance': {  # 执行当前语句以推进本节示例。
                'doc_id': doc_id, 'doc_version': doc_version,  # 执行当前语句以推进本节示例。
                'text_sha256': hashlib.sha256(text.encode()).hexdigest(),  # 执行当前语句以推进本节示例。
                'extractor_version': EXTRACTOR_VERSION,  # 执行当前语句以推进本节示例。
                'linker_version': LINKER_VERSION, 'kb_version': kb_version,  # 执行当前语句以推进本节示例。
                'tenant_id': tenant_id,  # 执行当前语句以推进本节示例。
            }  # 执行当前语句以推进本节示例。
        })  # 执行当前语句以推进本节示例。
    return outputs  # 返回当前分支计算出的结果。

linked = link_document('news-01', 2, 'tenant-a',  # 计算并保存当前步骤的中间状态。
                       '张伟医生在北大医院门诊，苹果公司发布 iPhone。', 3)  # 执行当前语句以推进本节示例。
assert any(row['entity_id'] == 'per:zhang-wei-01' and row['entity_type'] == 'PER' for row in linked)  # 用受控断言验证关键不变量。
assert any(row['entity_id'] == 'org:pku-hospital' and row['entity_type'] == 'ORG' for row in linked)  # 用受控断言验证关键不变量。
assert any(row['entity_id'] == 'company:apple' and row['entity_type'] == 'ORG' for row in linked)  # 用受控断言验证关键不变量。
print(json.dumps(linked, ensure_ascii=False, indent=2))  # 计算并保存当前步骤的中间状态。

## 9. 分层评估：span、candidate、link 各自负责

- span exact precision/recall/F1：边界与类型完全一致。
- relaxed/overlap 指标只用于错误分析，不能替代 exact 主指标。
- candidate recall@K：gold entity 是否进入候选。
- linking accuracy/F1：在 gold mention 上判断 entity_id。
- end-to-end entity F1：span + type + entity_id 全部正确。
- NIL/ABSTAIN：单独看拒识精确率、召回率和覆盖率。

必须按实体类型、mention 长度、是否别名、歧义度、语言、tenant、KB 新旧实体分桶。没有 gold 且没有预测的样本不提供正类 precision/recall 证据，下面返回 `None` 并从宏平均排除，避免大量空文档把 F1 人为抬到 1。

In [ ]:
def prf(predicted, gold):  # 定义本节可复用的核心函数。
    predicted, gold = set(predicted), set(gold)  # 计算并保存当前步骤的中间状态。
    tp = len(predicted & gold)  # 计算并保存当前步骤的中间状态。
    if not predicted and not gold:  # 按当前条件选择后续控制路径。
        return {'precision': None, 'recall': None, 'f1': None,  # 返回当前分支计算出的结果。
                'tp': 0, 'predicted': 0, 'gold': 0}  # 执行当前语句以推进本节示例。
    precision = tp / len(predicted) if predicted else 0.0  # 计算并保存当前步骤的中间状态。
    recall = tp / len(gold) if gold else None  # 计算并保存当前步骤的中间状态。
    f1 = (2*precision*recall/(precision+recall)  # 计算并保存当前步骤的中间状态。
          if recall is not None and precision+recall else 0.0 if recall is not None else None)  # 按当前条件选择后续控制路径。
    return {'precision': precision, 'recall': recall, 'f1': f1,  # 返回当前分支计算出的结果。
            'tp': tp, 'predicted': len(predicted), 'gold': len(gold)}  # 执行当前语句以推进本节示例。

gold_spans = {(0, 2, 'PER'), (6, 10, 'ORG')}  # 计算并保存当前步骤的中间状态。
pred_spans = {(0, 2, 'PER'), (6, 10, 'ORG'), (6, 8, 'ORG')}  # 计算并保存当前步骤的中间状态。
span_metrics = prf(pred_spans, gold_spans)  # 计算并保存当前步骤的中间状态。
gold_links = {(0, 2, 'per:zhang-wei-01'), (6, 10, 'org:pku-hospital')}  # 计算并保存当前步骤的中间状态。
pred_links = {(0, 2, 'per:zhang-wei-01'), (6, 10, 'org:pku-hospital')}  # 计算并保存当前步骤的中间状态。
link_metrics = prf(pred_links, gold_links)  # 计算并保存当前步骤的中间状态。
assert span_metrics['recall'] == 1.0 and span_metrics['precision'] < 1.0  # 用受控断言验证关键不变量。
assert link_metrics['f1'] == 1.0  # 用受控断言验证关键不变量。
assert prf(set(), set())['f1'] is None  # 用受控断言验证关键不变量。
print({'span_exact': span_metrics, 'entity_exact': link_metrics})  # 执行当前语句以推进本节示例。

## 10. 增量 KB：snapshot、幂等 upsert 与删除

KB 更新会改变 alias 候选数量和歧义分布。在线请求必须绑定一个可读 snapshot；索引、entity metadata 与 linker 特征必须来自同一 kb_version。

upsert 的幂等键可用 tenant + entity_id + version + payload hash。删除优先 tombstone，避免旧文档链接到被复用的 ID。别名冲突不是写入失败，而是需要显式记录并触发歧义评估。

In [ ]:
class VersionedKB:  # 定义承载本节状态与行为的数据结构。
    def __init__(self, rows):  # 定义本节可复用的核心函数。
        self.rows = {(e.tenant_id, e.entity_id, e.kb_version): e for e in rows}  # 计算并保存当前步骤的中间状态。
    def upsert(self, entity):  # 定义本节可复用的核心函数。
        key = (entity.tenant_id, entity.entity_id, entity.kb_version)  # 计算并保存当前步骤的中间状态。
        existing = self.rows.get(key)  # 计算并保存当前步骤的中间状态。
        if existing is not None and existing != entity:  # 按当前条件选择后续控制路径。
            raise ValueError('same_version_conflict')  # 遇到非法合同立即显式失败。
        older = [v for (t, eid, v) in self.rows if t == entity.tenant_id and eid == entity.entity_id]  # 计算并保存当前步骤的中间状态。
        if older and entity.kb_version < max(older):  # 按当前条件选择后续控制路径。
            raise ValueError('version_regression')  # 遇到非法合同立即显式失败。
        self.rows[key] = entity  # 计算并保存当前步骤的中间状态。
        return key  # 返回当前分支计算出的结果。
    def snapshot(self, tenant_id, version):  # 定义本节可复用的核心函数。
        return select_snapshot(self.rows.values(), tenant_id, version)  # 返回当前分支计算出的结果。

kb = VersionedKB(entities)  # 计算并保存当前步骤的中间状态。
new_entity = Entity('org:thu', 'tenant-a', '清华大学', ('清华', '清华大学'), 'ORG',  # 计算并保存当前步骤的中间状态。
                    ('高校', '大学', '科研'), True, 4)  # 执行当前语句以推进本节示例。
key1 = kb.upsert(new_entity)  # 计算并保存当前步骤的中间状态。
key2 = kb.upsert(new_entity)  # 计算并保存当前步骤的中间状态。
kb_rows_v4 = list(kb.rows.values())  # 计算并保存当前步骤的中间状态。
assert key1 == key2  # 用受控断言验证关键不变量。
assert any(e.entity_id == 'org:thu' for e in kb.snapshot('tenant-a', 4))  # 用受控断言验证关键不变量。
assert any(e.entity_id == 'org:pku' for e in kb.snapshot('tenant-a', 4))  # 用受控断言验证关键不变量。
assert all(e.entity_id != 'org:old' for e in kb.snapshot('tenant-a', 4))  # 用受控断言验证关键不变量。
assert [e.entity_id for e in generate_candidates('北大', 'tenant-a', 4, entity_rows=kb_rows_v4)] == ['org:pku']  # 用受控断言验证关键不变量。
linked_v4 = link_document('news-v4', 1, 'tenant-a', '北大和清华大学开展科研合作', 4,  # 计算并保存当前步骤的中间状态。
                          entity_rows=kb_rows_v4)  # 计算并保存当前步骤的中间状态。
assert {row['entity_id'] for row in linked_v4} == {'org:pku', 'org:thu'}  # 用受控断言验证关键不变量。
assert all(row['provenance']['kb_version'] == 4 for row in linked_v4)  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    kb.upsert(Entity('org:thu', 'tenant-a', '冲突名称', ('清华',), 'ORG', (), True, 4))  # 执行当前语句以推进本节示例。
    raise AssertionError('同版本冲突必须失败')  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert str(exc) == 'same_version_conflict'  # 用受控断言验证关键不变量。
print([e.entity_id for e in kb.snapshot('tenant-a', 4)])  # 执行当前语句以推进本节示例。

## 11. 典型失败与定位顺序

1. span 漏召回：别名缺失、tokenizer/规范化或嵌套策略问题。
2. gold 不在 candidate：tenant/kb_version 过滤错误，或 alias 索引未增量更新。
3. candidate 有 gold 但排错：上下文窗口、类型或流行度特征偏置。
4. top1/top2 接近仍强行链接：缺少拒识，NIL 精度下降。
5. 指标异常升高：同一实体描述或同文档族泄漏到 train/test。
6. 线上 offset 漂移：调用端把 UTF-16、UTF-8 byte 与 code point 混用。
7. KB 删除后仍返回：缓存与向量/搜索索引未随 snapshot 失效。

In [ ]:
# 关键合同回归
assert spans_to_bilou('北大', [(0, 2, 'ORG')]) == ['B-ORG', 'L-ORG']  # 用受控断言验证关键不变量。
try:  # 尝试执行可能失败的受控操作。
    spans_to_bilou('北大医院', [(0, 2, 'ORG'), (0, 4, 'ORG')])  # 执行当前语句以推进本节示例。
    raise AssertionError('重叠 span 必须显式分层')  # 遇到非法合同立即显式失败。
except ValueError as exc:  # 捕获预期异常并验证失败分支。
    assert 'overlapping' in str(exc)  # 用受控断言验证关键不变量。
assert codepoint_to_utf8_span('😀北大', 1, 3) == (4, 10)  # 用受控断言验证关键不变量。
assert [m['surface'] for m in longest_leftmost(  # 用受控断言验证关键不变量。
    find_all_mentions('北大医院', alias_a))] == ['北大医院']  # 计算并保存当前步骤的中间状态。
assert german_mentions[0]['surface'] == 'Straße' and german_mentions[0]['end'] == 8  # 用受控断言验证关键不变量。
assert len(generate_candidates('苹果', 'tenant-a', 3)) == 2  # 用受控断言验证关键不变量。
assert generate_candidates('苹果', 'tenant-b', 8) == []  # 用受控断言验证关键不变量。
assert [e.entity_id for e in generate_candidates('北大', 'tenant-a', 4,  # 用受控断言验证关键不变量。
                                                  entity_rows=kb_rows_v4)] == ['org:pku']  # 计算并保存当前步骤的中间状态。
assert choose_or_abstain(rank_candidates('苹果', '甜水果', apple_candidates))['entity_id'] == 'concept:apple-fruit'  # 用受控断言验证关键不变量。
assert choose_or_abstain(rank_candidates('苹果', '没有上下文', apple_candidates))['entity_id'] is None  # 用受控断言验证关键不变量。
assert all(row['provenance']['kb_version'] == 3 for row in linked)  # 用受控断言验证关键不变量。
assert all(row['provenance']['tenant_id'] == 'tenant-a' for row in linked)  # 用受控断言验证关键不变量。
assert all(row['entity_type'] in {'PER', 'ORG', 'PRODUCT'} for row in linked)  # 用受控断言验证关键不变量。
assert all(row['surface'] == '张伟医生在北大医院门诊，苹果公司发布 iPhone。'[row['start']:row['end']]  # 用受控断言验证关键不变量。
           for row in linked)  # 遍历输入元素以累积或检查结果。
assert prf(set(), set())['f1'] is None  # 用受控断言验证关键不变量。
print('NER + Entity Linking 合同回归：全部通过')  # 执行当前语句以推进本节示例。

## 12. 服务、批处理与安全

在线接口建议：

POST /v1/entity-link
请求携带 doc_id/doc_version/text/kb_version；tenant 从认证上下文注入。
响应携带 mentions、offset_unit、text_sha256、model/index versions、trace_id、degraded 标记。

批处理要按 token/字符预算组 batch，而不是固定文档数；长文档切窗时保留 global offset，并去重跨窗 mention。缓存键包含 tenant、text hash、extractor/linker/kb version。日志默认只记 hash、长度、候选数和 reason，不记全文。对 candidate/entity metadata 的访问也做 ACL，不能在 ranker 后再过滤。

## 13. 生产替换点

- 规则 NER → 经过领域数据评估的 CRF、BiLSTM-CRF 或 Transformer span/token classifier；保留词典作为高精度通道。
- 线性上下文命中 → bi-encoder 候选检索 + cross-encoder/图特征重排，并做概率校准。
- 小字典全扫描 → Trie/Aho–Corasick 或搜索索引。
- 单机 dict KB → 带 snapshot/CDC 的实体服务与搜索/向量索引，原子发布版本。
- longest-leftmost → 按任务选择嵌套 span model、weighted interval scheduling 或多层输出。

延迟预算要拆为 NER、candidate、feature fetch、ranker、KB；任一路超时必须返回 degraded 原因。模型升级时固定 candidate set 做 ranker 对比，再做端到端 shadow，避免召回变化掩盖排序回归。

## 14. 原论文与官方资料

1. Tjong Kim Sang & De Meulder, CoNLL-2003 Shared Task: Language-Independent NER：https://aclanthology.org/W03-0419/
2. Ratinov & Roth, Design Challenges and Misconceptions in Named Entity Recognition（讨论 BILOU）：https://aclanthology.org/W09-1119/
3. Hoffart et al., Robust Disambiguation of Named Entities in Text（AIDA）：https://aclanthology.org/D11-1072/
4. Shen, Wang, Han, Entity Linking with a Knowledge Base: Issues, Techniques, and Solutions：https://doi.org/10.1109/TKDE.2014.2327028
5. Unicode Standard Annex #29, Text Segmentation：https://unicode.org/reports/tr29/
6. Unicode CaseFolding 数据与状态定义：https://www.unicode.org/Public/UCD/latest/ucd/CaseFolding.txt

技术边界：词典覆盖率、上下文关键词和阈值都是受控教学设定；未在真实长尾实体、跨语种、嵌套实体及大规模 KB 上验证，不能据此声明通用 NER/EL 精度。